# Probabilidad

El capítulo anterior describió **datos que ya pasaron**. Acá pasamos a hablar
de **incertidumbre sobre lo que viene**: si llega un paciente, ¿qué tan
probable es que espere más de 5 minutos? Si una persona entra a la encuesta,
¿qué tan probable es que diga «sí»?

El idioma de esta unidad son las **operaciones de eventos** (uniones,
intersecciones, complementos), la **probabilidad condicional** y el
**teorema de Bayes**. Vamos a usarlo para preguntas que ya empezaron a aparecer:
qué tan probable es que la espera supere cierto umbral, qué tan creíble es un
resultado positivo en un test diagnóstico (la misma forma de razonar que se
aplica a una encuesta de intención de voto) y qué tan probable es que una pieza
defectuosa pase la inspección.


## Preguntas de inicio

1. ¿Cómo combinamos eventos? Dado «espera mucho» y «paciente con turno», ¿cuál
   es la probabilidad de la unión y de la intersección?
2. Si **ya** sabemos algo (el paciente esperó 3 minutos), ¿cómo cambia la
   probabilidad de los próximos minutos?
3. Un test diagnóstico tiene 99% de sensibilidad y 95% de especificidad. Si da
   positivo, ¿qué tan creíble es?
4. ¿Por qué un test que parece buenísimo casi no mueve la creencia inicial?


## Importaciones

In [ ]:
from core import Settings
from exercises import NumericAnswerInput, verify_numeric_answer
from probability import (
    BayesInput,
    ConditionalInput,
    JointEventInput,
    SetOperationInput,
    evaluate_bayes,
    evaluate_conditional_probability,
    evaluate_set_operations,
    joint_event_probabilities,
)
from probability.total_probability import TotalProbabilityBranch
from symbolic import bayes_theorem, total_probability_theorem
from widgets import BayesExplorerInput, build_bayes_explorer

In [ ]:
settings = Settings()

## Un experimento concreto

Antes de hablar de tiempos continuos, fijamos un experimento minimalista: tirar
un dado equilibrado. Definimos dos eventos:

- $A$ = «sale par» = $\{2, 4, 6\}$.
- $B$ = «sale al menos 4» = $\{4, 5, 6\}$.

Sobre este universo de 6 resultados podemos visualizar las operaciones de
conjuntos sin ambigüedad. Después transferimos la idea a eventos sobre
tiempos o resultados de la encuesta.


In [ ]:
universe = frozenset({"1", "2", "3", "4", "5", "6"})
event_even = frozenset({"2", "4", "6"})
event_at_least_four = frozenset({"4", "5", "6"})

set_input = SetOperationInput(
    universe=universe,
    event_a=event_even,
    event_b=event_at_least_four,
)
set_result = evaluate_set_operations(set_input)
set_result

## La regla aditiva

**Paso 1 — Cardinalidades.** Imaginá dos círculos solapados (Venn): $A \cap B$
vive en la zona compartida. Si sumamos cardinalidades a lo bestia, la zona
compartida se cuenta dos veces:

$$ |A \cup B| = |A| + |B| - |A \cap B| \tag{2.1} $$

**Paso 2 — Probabilidades.** Dividiendo cada cardinalidad por $|\Omega|$
obtenemos la **regla aditiva** sobre probabilidades. Esta es la primera
fórmula propiamente probabilística del libro:

$$ P(A \cup B) = P(A) + P(B) - P(A \cap B) \tag{2.2} $$

Volviendo al dado: $P(A) = P(B) = 3/6$, $P(A \cap B) = 2/6$, así que la
ecuación (2.2) da $P(A \cup B) = 3/6 + 3/6 - 2/6 = 4/6$.


In [ ]:
joint_input = JointEventInput(
    probability_a=3 / 6,
    probability_b=3 / 6,
    probability_intersection=2 / 6,
)
joint = joint_event_probabilities(joint_input)
joint

## Probabilidad condicional

Probabilidad condicional es **restringir el universo**: dentro del subconjunto
$B$, ¿qué fracción cae también en $A$? **Paso 1:** retomamos $P(A \cap B)$ que
ya definimos junto a (2.2). **Paso 2:** definimos:

$$ P(A \mid B) = \frac{P(A \cap B)}{P(B)},\quad P(B) > 0 \tag{2.3} $$

Con los datos del dado: $P(A \mid B) = (2/6)/(3/6) = 2/3$. Saber que el
resultado es al menos 4 sube la chance de «par» de $1/2$ a $2/3$.


In [ ]:
conditional_input = ConditionalInput(
    probability_intersection=2 / 6,
    probability_conditioning_event=3 / 6,
)
conditional = evaluate_conditional_probability(conditional_input)
conditional

## Las esperas vistas como eventos

Volvemos a la sala de espera y miramos la fórmula (2.3) en ese contexto. Tenemos
$T$ = «espera más de 5 minutos» y $S$ = «espera más de 3 minutos». Por
construcción $T \subset S$, así que $P(T \cap S) = P(T)$. La fórmula (2.3)
queda:

$$ \begin{aligned}
P(T \mid S) &= \frac{P(T \cap S)}{P(S)} \\[4pt]
            &= \frac{P(T)}{P(S)}
\end{aligned} $$

Si en una semana de datos pasados $P(T) = 0{,}21$ y $P(S) = 0{,}45$,
entonces $P(T \mid S) = 0{,}21 / 0{,}45 \approx 0{,}47$. Ya esperar 3 minutos
casi duplica la chance original de cruzar los 5.


In [ ]:
clinic_conditional_input = ConditionalInput(
    probability_intersection=0.21,
    probability_conditioning_event=0.45,
)
clinic_conditional = evaluate_conditional_probability(clinic_conditional_input)
clinic_conditional

## Teorema de Bayes (forma simbólica)

**Paso 1.** Escribimos la condicional en los dos sentidos usando (2.3):

$$ P(A \cap B) = P(A \mid B)\,P(B) = P(B \mid A)\,P(A) $$

**Paso 2.** Igualamos los dos miembros de la derecha y despejamos
$P(A \mid B)$:

$$ P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)} \tag{2.4} $$

El módulo `symbolic` mantiene esa forma simbólica para reutilizarla:


In [ ]:
bayes_theorem().formula

## Bayes con datos: prueba diagnóstica

**Concreto.** En una población, $P(D) = 0{,}01$ es la prevalencia. El test
tiene sensibilidad $P(+ \mid D) = 0{,}99$ y especificidad
$P(- \mid \bar{D}) = 0{,}95$ (ergo $P(+ \mid \bar{D}) = 0{,}05$). Buscamos
$P(D \mid +)$.

**Pictórico.** Pensá en 10.000 personas. 100 enfermos: 99 dan positivo.
9.900 sanos: 495 dan positivo. Total de positivos: $99 + 495 = 594$. La
fracción de verdaderos enfermos dentro de los positivos es
$99 / 594 \approx 0{,}167$.

**Abstracto.** Aplicamos (2.4) con la **probabilidad total** en el
denominador. Para no sufrir scroll horizontal, expresamos la cuenta en
varias líneas:

$$ \begin{aligned}
P(D \mid +) &= \frac{P(+ \mid D)\,P(D)}{P(+)} \\[4pt]
P(+)        &= P(+ \mid D)\,P(D) + P(+ \mid \bar{D})\,P(\bar{D}) \\[4pt]
            &= 0{,}99 \cdot 0{,}01 + 0{,}05 \cdot 0{,}99 \\[4pt]
            &= 0{,}0099 + 0{,}0495 \\[4pt]
            &= 0{,}0594 \\[4pt]
P(D \mid +) &= \frac{0{,}0099}{0{,}0594} \approx 0{,}167
\end{aligned} $$


In [ ]:
diagnostic_branches = (
    TotalProbabilityBranch(label="Enfermo", prior=0.01, likelihood=0.99),
    TotalProbabilityBranch(label="Sano", prior=0.99, likelihood=0.05),
)
diagnostic_input = BayesInput(branches=diagnostic_branches)
diagnostic_posteriors = evaluate_bayes(diagnostic_input)
diagnostic_posteriors[0]

Aunque el test es excelente, un positivo apenas mueve la creencia del 1% al
16,7%. La culpa la tiene la **tasa base**: como casi nadie está enfermo, los
falsos positivos pesan mucho. Movés los sliders del widget y vas a ver cómo
el posterior reacciona.


In [ ]:
explorer_input = BayesExplorerInput(settings=settings)
build_bayes_explorer(explorer_input)

## Probabilidad total — la versión general

Si $\{A_1, \dots, A_k\}$ es una **partición** del universo (eventos
incompatibles que cubren todo), la probabilidad total da el denominador
limpio que usamos en el ejemplo anterior:

$$ P(B) = \sum_{i=1}^{k} P(B \mid A_i)\,P(A_i) \tag{2.5} $$


In [ ]:
total_probability_theorem(partition_size=3).formula

## Una pieza pasa la inspección

La línea de producción inspecciona cada pieza con un test que detecta
defectos con probabilidad $0{,}9$ si la pieza está mal y produce un falso
positivo con probabilidad $0{,}05$ si está bien. La verdadera tasa de piezas
defectuosas es $0{,}03$.

Aplicamos (2.4) y (2.5) para responder: **dada una alarma, ¿cuál es la
probabilidad de que la pieza esté realmente defectuosa?**


In [ ]:
factory_branches = (
    TotalProbabilityBranch(label="Defectuosa", prior=0.03, likelihood=0.90),
    TotalProbabilityBranch(label="Sana", prior=0.97, likelihood=0.05),
)
factory_input = BayesInput(branches=factory_branches)
factory_posteriors = evaluate_bayes(factory_input)
factory_posteriors[0]

## Ejercicio 1 — Regla aditiva

Sean $P(A) = 0{,}6$, $P(B) = 0{,}5$ y $P(A \cap B) = 0{,}2$. Calculá
$P(A \cup B)$ usando la fórmula (2.2):

$$ P(A \cup B) = 0{,}6 + 0{,}5 - 0{,}2 = 0{,}9 $$


In [ ]:
exercise_joint_input = JointEventInput(
    probability_a=0.6,
    probability_b=0.5,
    probability_intersection=0.2,
)
expected_union = joint_event_probabilities(exercise_joint_input).union

student_answer_union = 0.9
verify_input = NumericAnswerInput(
    student_answer=student_answer_union,
    expected_answer=expected_union,
)
verify_numeric_answer(verify_input)

## Ejercicio 2 — Bayes a mano

Una caja $C_1$ tiene 3 bolas rojas y 7 blancas. La caja $C_2$ tiene 6 rojas
y 4 blancas. Se elige una caja al azar y se saca una bola: resulta roja.
Calculá $P(C_1 \mid R)$ aplicando la fórmula (2.4) con denominador (2.5):

$$ \begin{aligned}
P(C_1 \mid R) &= \frac{P(R \mid C_1)\,P(C_1)}{P(R)} \\[4pt]
P(R)          &= 0{,}3 \cdot 0{,}5 + 0{,}6 \cdot 0{,}5 \\[4pt]
              &= 0{,}45 \\[4pt]
P(C_1 \mid R) &= \frac{0{,}3 \cdot 0{,}5}{0{,}45} \\[4pt]
              &= \frac{1}{3}
\end{aligned} $$


In [ ]:
box_branches = (
    TotalProbabilityBranch(label="C1", prior=0.5, likelihood=0.3),
    TotalProbabilityBranch(label="C2", prior=0.5, likelihood=0.6),
)
box_input = BayesInput(branches=box_branches)
expected_posterior = evaluate_bayes(box_input)[0].posterior

student_answer_posterior = 1 / 3
verify_input = NumericAnswerInput(
    student_answer=student_answer_posterior,
    expected_answer=expected_posterior,
)
verify_numeric_answer(verify_input)

## Síntesis y respuestas

1. **¿Cómo combinamos eventos?** La regla aditiva (2.2) muestra que
   $P(A \cup B) = P(A) + P(B) - P(A \cap B)$, restando la intersección para
   no contarla dos veces.
2. **¿Cómo cambia una probabilidad si ya sabemos algo?** Por la fórmula (2.3),
   $P(A \mid B)$ restringe el universo a $B$ y mide qué fracción cae también
   en $A$.
3. **¿Qué tan creíble es un test positivo?** Por (2.4) y (2.5), $P(D \mid +)$
   depende de la **prevalencia**. Con prevalencia 1%, un test 99/95 lleva la
   creencia a 16,7% — no a 99%.
4. **¿Por qué un test bueno mueve poco la creencia?** Porque la masa de la
   probabilidad total $P(+)$ proviene de los falsos positivos cuando la tasa
   base es chica. Es la misma cuenta de la línea de inspección con piezas defectuosas.


## Próximas preguntas

Hasta acá los eventos eran discretos. En el próximo capítulo:

- ¿Cómo modelamos «el tiempo de espera» como una **variable aleatoria** con
  una densidad?
- ¿Qué distribución usar para contar piezas defectuosas en un turno?
- ¿Y para la altura de las personas que vienen a la clínica?

Con la maquinaria de variables aleatorias vamos a hablar de probabilidades de
intervalos sin tener que enumerar resultados.
